In [1]:
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset,DataLoader

In [8]:
#datasets
import torchvision
from torchvision import datasets,transforms

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,))
])

trainset=datasets.MNIST(root="./data",download=True,train=True,transform=transform)
testset=datasets.MNIST(root="./data",download=True,train=False,transform=transform)

In [9]:
trainloader=DataLoader(trainset,shuffle=True,batch_size=64)
testloader=DataLoader(testset,batch_size=64)

In [10]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_layers=nn.Sequential(
            nn.Conv2d(1,28,padding=1,kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #14x14

            nn.Conv2d(28,56,padding=1,kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #7x7

        )

        self.fc_layers=nn.Sequential(
            nn.Flatten(),
            nn.Linear(56*7*7,128),
            nn.ReLU(),
            nn.Linear(128,10)
        )

    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layers(x)

        return x

In [11]:
model=CNN()

In [12]:
criteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [13]:
epochs=10
for epoch in range(epochs):
    training_loss=0.0

    for images,labels in trainloader:
        optimizer.zero_grad()

        output=model.forward(images)
        loss=criteria(output,labels)
        loss.backward()
        optimizer.step()

        training_loss+=loss.item()

    print(f"for {epoch+1}/{epochs} & loss={training_loss/len(trainloader)}")

for 1/10 & loss=0.15486953198772743
for 2/10 & loss=0.04682282410099852
for 3/10 & loss=0.03211997146023187
for 4/10 & loss=0.02296171386406679
for 5/10 & loss=0.018309366338931498
for 6/10 & loss=0.014941416884772355
for 7/10 & loss=0.011769322796807131
for 8/10 & loss=0.009623873113774445
for 9/10 & loss=0.007583221266896992
for 10/10 & loss=0.006700413874373148


In [14]:
import torch
model.eval()
correct=0.0
total=0.0
with torch.no_grad():
    for images,labels in testloader:
        output=model.forward(images)
        _,predicted=torch.max(output,1)
    
        correct+=(predicted==labels).sum().item()
        total+=labels.size(0)

    print(f"accuracy={correct/total *100}")


accuracy=98.89
